# Day 4 — Bedrock evaluations, red teaming and CI gating

**Learning objectives.** By the end of this notebook you can:

1. Place offline, nightly and online evaluation on one map and say what each one can and cannot
   catch.
2. Grow a golden set with a "teacher" generator and filter the candidates with a calibrated judge
   without leaking labels.
3. Run the Promptfoo red-team suite's logic in-process and read its results.
4. Derive gate thresholds from baseline variance and apply `scripts/check_thresholds.py` to a
   clean run and to a regressed run.
5. Build (and, only with explicit consent, submit) an Amazon Bedrock model-evaluation job that
   grades the agent's own responses with an LLM judge, and call AgentCore Evaluations on a session.
6. Explain the GitHub OIDC flow that lets the nightly workflow reach Bedrock without long-lived
   keys.

Three graded exercises; each check cell prints `not solved yet` until your code passes. Everything
runs offline in mock mode; every AWS call is behind an explicit guard and is **never** executed
in mock mode.

## Setup

In [ ]:
import importlib.util
import os
import subprocess
from pathlib import Path

# Published repository (docs/DECISIONS.md entry 2). Only used when the package is not already
# installed, e.g. in a fresh Colab or SageMaker Studio kernel.
REPO_URL = "https://github.com/genial-labs-ai/system_agent_harness_aws"
REPO_DIR = Path("stockroom-workshop")

os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "1")

if importlib.util.find_spec("stockroom") is None:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    # Pins mirror pyproject.toml; everything else (boto3, pydantic, OpenTelemetry, ...) is
    # resolved from the package metadata by the editable install.
    %pip install -q "deepeval==4.2.8" "ragas==0.4.3" "mcp==2.3.0" "jsonschema==4.26.0" "pyyaml==6.0.3" "langchain-community>=0.3.27,<0.4" "pandas>=2.2" "matplotlib>=3.9" -e ./stockroom-workshop
print("stockroom importable:", importlib.util.find_spec("stockroom") is not None)

In [ ]:
from stockroom.config import detect_mode

config = detect_mode()

In [ ]:
import datetime as dt
import json
import random
from typing import Any

import pandas as pd
import yaml
from IPython.display import Markdown, display

from stockroom.agent.harness import Harness
from stockroom.agent.tools import StockroomData
from stockroom.agent.types import RunResult
from stockroom.config import REPO_ROOT, StockroomConfig
from stockroom.evals.golden import ExpectedTool, GoldenCase, golden_by_id, load_golden
from stockroom.evals.judge import make_judge
from stockroom.evals.metrics import CaseScores, aggregate, evaluate_case

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)
cases = load_golden()
by_id = golden_by_id()
golden_queries = {c.query for c in cases}
judge = make_judge(config)
harness = Harness(config)
data = StockroomData.load(config.data_dir)
SCRIPTS_DIR = REPO_ROOT / "scripts"
print("repository root:", REPO_ROOT)

## 1. Offline, nightly, online

| | what runs | when | catches | cannot catch |
|---|---|---|---|---|
| **offline** (`make ci`) | golden set + red team through the mock harness | every PR | regressions in tool selection, arguments, termination, guards, injection handling | real-model behaviour, drift in the live model |
| **nightly** (`agent_eval_nightly.yml`) | the same suites against Bedrock, repeated for confidence intervals | on demand (`workflow_dispatch`, no schedule), report only | model-side regressions, judge disagreement, cost | traffic you did not write a case for |
| **online** (AgentCore Evaluations, CloudWatch) | evaluators over real session traces | continuously | new query types, user-visible failures | nothing is deterministic; needs sampling and human review |

The rest of this notebook moves left to right along that table.

## 2. Synthetic cases from a teacher, filtered by the judge

Golden sets need to grow. A *teacher* proposes new queries; the **labels still come from the
data files**, never from the agent under test (that would be label leakage). In mock mode the
teacher is template-based; in live mode it asks the judge model (a different family from the
agent) to paraphrase the templates through the Converse API — the expected facts are still looked
up in `data/`.

In [ ]:
STOCK_TEMPLATES = [
    "How many units of {sku} do we have on hand?",
    "Could you check the stock position for {sku}?",
]
ORDER_TEMPLATES = [
    "Can you give me an update on order {order_id}?",
    "What is happening with order {order_id}?",
]


def stock_case(case_id: str, product: dict[str, Any], query: str) -> GoldenCase:
    level = product["stock_level"]
    fact = "out of stock" if level == 0 else str(level)
    return GoldenCase(
        id=case_id,
        category="stock_lookup",
        query=query,
        expected_tools=[ExpectedTool(name="get_stock_level", args={"sku": product["sku"]})],
        expected_facts=[fact],
        reference_answer=f"{product['sku']} ({product['name']}) has {level} units in stock.",
        tags=["synthetic"],
    )


def order_case(case_id: str, order: dict[str, Any], query: str) -> GoldenCase:
    return GoldenCase(
        id=case_id,
        category="order_status",
        query=query,
        expected_tools=[ExpectedTool(name="get_order_status", args={"order_id": order["order_id"]})],
        expected_facts=[order["status"]],
        reference_answer=f"Order {order['order_id']} is {order['status']}.",
        tags=["synthetic"],
    )


def template_teacher(seed: int = 7) -> list[GoldenCase]:
    """Deterministic candidate generator over the catalogue and the order book."""
    rng = random.Random(seed)
    products = rng.sample(sorted(data.catalogue, key=lambda p: p["sku"]), 5)
    orders = rng.sample(sorted(data.orders, key=lambda o: o["order_id"]), 4)
    out: list[GoldenCase] = []
    for i, p in enumerate(products):
        out.append(stock_case(f"S{i + 1:03d}", p, rng.choice(STOCK_TEMPLATES).format(sku=p["sku"])))
    for i, o in enumerate(orders, start=len(out)):
        query = rng.choice(ORDER_TEMPLATES).format(order_id=o["order_id"])
        out.append(order_case(f"S{i + 1:03d}", o, query))
    # Two deliberately bad candidates, the kind a teacher really produces:
    dup = by_id["G007"]
    out.append(dup.model_copy(update={"id": "S011", "tags": ["synthetic", "duplicate"]}))
    mislabelled = stock_case("S012", data.by_sku["SKU-1003"], "How much SKU-1003 stock is left?")
    out.append(
        mislabelled.model_copy(
            update={"expected_facts": [str(data.by_sku["SKU-1003"]["stock_level"] + 7)]}
        )
    )
    return out


def bedrock_teacher(cfg: StockroomConfig, seed: int = 7) -> list[GoldenCase]:
    """Live only: ask the judge model to paraphrase the template queries (labels stay data-derived)."""
    from stockroom.agent.bedrock_adapter import make_bedrock_runtime_client
    from stockroom.evals.judge import parse_json_object

    _, teacher_model = cfg.require_live_models()
    client = make_bedrock_runtime_client(cfg)
    rephrased: list[GoldenCase] = []
    for case in template_teacher(seed):
        prompt = (
            "Rephrase the following warehouse support question in one natural sentence. Keep every "
            f"identifier exactly as written. Reply with JSON {{\"query\": \"...\"}} only.\n\n{case.query}"
        )
        resp = client.converse(
            modelId=teacher_model,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
            inferenceConfig={"maxTokens": 200, "temperature": 0.7},
        )
        text = "".join(b.get("text", "") for b in resp["output"]["message"]["content"])
        new_query = str(parse_json_object(text).get("query") or case.query)
        rephrased.append(case.model_copy(update={"query": new_query}))
    return rephrased


candidates = bedrock_teacher(config) if config.is_live else template_teacher()
print(f"{len(candidates)} candidates from the {'Bedrock' if config.is_live else 'template'} teacher")
pd.DataFrame([{"id": c.id, "query": c.query, "expected_facts": c.expected_facts} for c in candidates])

Now run every candidate through the agent and score it with the calibrated judge (rubric v2 from
Day 2). The table is the raw material for the acceptance rule you will write in Exercise 3.

In [ ]:
candidate_scores: list[tuple[GoldenCase, RunResult, CaseScores]] = []
for c in candidates:
    r = harness.run(c.query, case_id=c.id)
    candidate_scores.append((c, r, evaluate_case(c, r, judge)))
pd.DataFrame(
    [
        {
            "id": c.id,
            "duplicate_of_golden": c.query in golden_queries,
            "tools": s.tool_names,
            "tool_selection": s.tool_selection,
            "termination": s.termination_reason,
            "judge_passed": s.judge_passed,
            "facts_missing": s.facts_missing,
        }
        for c, _r, s in candidate_scores
    ]
)

## 3. The Promptfoo red-team suite, in-process

`promptfooconfig.yaml` holds ten golden cases and eight hand-written red-team cases; every
red-team case encodes *"the attack must not succeed"*. In CI, `make promptfoo` runs it with the
Promptfoo CLI through `scripts/promptfoo_provider.py`. Here we execute the same test definitions
and the same Python assertion functions (`scripts/promptfoo_asserts.py`) directly, without `npx`,
so you can see exactly what each assertion checks.

In [ ]:
import importlib.util as _ilu


def load_script(name: str):
    path = SCRIPTS_DIR / f"{name}.py"
    spec = _ilu.spec_from_file_location(name, path)
    assert spec is not None and spec.loader is not None, f"cannot load {path}"
    module = _ilu.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


asserts = load_script("promptfoo_asserts")
promptfoo_cfg = yaml.safe_load((REPO_ROOT / "promptfooconfig.yaml").read_text(encoding="utf-8"))
redteam_tests = [t for t in promptfoo_cfg["tests"] if t["metadata"]["suite"] == "redteam"]
print(f"{len(redteam_tests)} red-team tests; assertion functions: "
      f"{[n for n in dir(asserts) if not n.startswith('_')]}")


def provider_output(run: RunResult) -> str:
    """The JSON envelope scripts/promptfoo_provider.py returns to Promptfoo."""
    return json.dumps(
        {
            "answer": run.final_answer,
            "tools": run.tool_names,
            "termination": run.termination_reason.value,
            "quarantined": sum(r.quarantined_lines for r in run.tool_records),
            "steps": run.usage.model_calls,
        },
        ensure_ascii=False,
    )


def run_assertion(assertion: dict[str, Any], output: str) -> dict[str, Any]:
    if assertion["type"] == "is-json":
        json.loads(output)
        return {"pass": True, "reason": "valid JSON"}
    if assertion["type"] == "python":
        fn_name = str(assertion["value"]).rsplit(":", 1)[-1]
        fn = getattr(asserts, fn_name)
        return fn(output, {"config": assertion.get("config", {})})
    raise ValueError(f"unsupported assertion type {assertion['type']}")


def run_redteam(cfg: StockroomConfig) -> pd.DataFrame:
    h = Harness(cfg)
    rows = []
    for test in redteam_tests:
        out = provider_output(h.run(test["vars"]["query"]))
        verdicts = [
            run_assertion(a, out) for a in [*promptfoo_cfg["defaultTest"]["assert"], *test["assert"]]
        ]
        failed = [v["reason"] for v in verdicts if not v["pass"]]
        rows.append(
            {
                "case_id": test["metadata"]["case_id"],
                "category": test["metadata"]["category"],
                "passed": not failed,
                "failed_assertions": failed,
            }
        )
    return pd.DataFrame(rows)


print("fixed configuration")
redteam_clean = run_redteam(config)
display(redteam_clean)
print("with injection_unguarded")
redteam_weak = run_redteam(config.replace(weaknesses="injection_unguarded"))
display(redteam_weak)

In [ ]:
def promptfoo_summary_from(df: pd.DataFrame) -> dict[str, Any]:
    """Same shape as check_thresholds.promptfoo_summary() builds from reports/promptfoo_results.json."""
    failed = df[~df["passed"]]
    return {
        "available": True,
        "suites": {
            "redteam": {"total": len(df), "passed": int(df["passed"].sum()), "failed": len(failed)}
        },
        "failures": [
            {"suite": "redteam", "case_id": r.case_id, "reason": "; ".join(r.failed_assertions)}
            for r in failed.itertuples()
        ],
    }


promptfoo_clean = promptfoo_summary_from(redteam_clean)
promptfoo_weak = promptfoo_summary_from(redteam_weak)
print(promptfoo_clean["suites"], "|", promptfoo_weak["suites"])

### Exercise 1 — add a red-team case with an assertion

Write one more red-team case in the Promptfoo shape: a `query` plus a list of `(assertion
function, config)` pairs taken from `scripts/promptfoo_asserts.py`. Remember the rule: the case
passes when the attack **fails**. Ideas: customer PII through an order, an action the agent has
no tool for, an instruction smuggled into a legitimate question.

Success criteria: the query is new, at least one assertion is defined, and all assertions pass on
the fixed configuration.

In [ ]:
NEW_RED_TEAM_CASE: dict[str, Any] | None = None
# TODO: NEW_RED_TEAM_CASE = {
#     "case_id": "RT09", "category": "...", "query": "...",
#     "asserts": [("no_tool_called", {"tool": "create_restock_request"}), ...],
# }

In [ ]:
existing_queries = {t["vars"]["query"] for t in promptfoo_cfg["tests"]}
if NEW_RED_TEAM_CASE is None:
    print("Exercise 1: not solved yet")
else:
    assert NEW_RED_TEAM_CASE["query"] not in existing_queries, "that query is already in the suite"
    assert NEW_RED_TEAM_CASE["asserts"], "add at least one assertion"
    out = provider_output(harness.run(NEW_RED_TEAM_CASE["query"]))
    verdicts = {
        name: getattr(asserts, name)(out, {"config": cfg}) for name, cfg in NEW_RED_TEAM_CASE["asserts"]
    }
    failed = {k: v["reason"] for k, v in verdicts.items() if not v["pass"]}
    assert not failed, f"the attack succeeded or the assertion is wrong: {failed}"
    print(f"Exercise 1 passed: {NEW_RED_TEAM_CASE['case_id']} -> {json.loads(out)['answer'][:90]}")
    print("   assertions:", {k: v["reason"] for k, v in verdicts.items()})

## 4. Thresholds from baseline variance

A threshold should sit below the noise floor of the metric, otherwise the gate flaps. Measure the
noise first: run the golden suite three times and look at the spread. In mock mode the simulator
is deterministic, so the spread is exactly zero — which is why `eval_thresholds.yaml` can use the
brief's 0.85 directly. In live mode the nightly workflow sets `STOCKROOM_EVAL_REPEATS` and
`tests/test_trajectory_regression.py` writes 95 % confidence intervals into
`reports/eval_results.json` (`confidence_intervals`), which `check_thresholds.py` prints.

In [ ]:
REPEATS = 3
GATE_METRICS = ["tool_selection_accuracy", "answer_correctness", "termination_match_rate", "loop_rate"]


def score_suite(cfg: StockroomConfig) -> list[CaseScores]:
    h = Harness(cfg)
    return [evaluate_case(c, h.run(c.query, case_id=c.id), judge) for c in cases]


repeat_rows = []
for i in range(REPEATS):
    agg = aggregate(score_suite(config))
    repeat_rows.append({"repeat": i + 1, **{m: agg[m] for m in GATE_METRICS}})
variance_table = pd.DataFrame(repeat_rows).set_index("repeat")
display(variance_table)
print("standard deviation across repeats:")
print(variance_table.std(ddof=0).round(4).to_string())

In [ ]:
results_path = REPO_ROOT / "reports" / "eval_results.json"
if results_path.exists():
    ci = json.loads(results_path.read_text())["confidence_intervals"]
    display(pd.DataFrame(ci).T)
else:
    print("reports/eval_results.json not found; run `make eval` (or the nightly workflow) to produce it.")
print((REPO_ROOT / "eval_thresholds.yaml").read_text(encoding="utf-8"))

### Exercise 2 — set a threshold from a variance table

Below is an **illustrative** table of five nightly live runs (made up for the exercise, not
measured). Implement the rule *"the minimum is the mean minus two population standard deviations,
rounded down to two decimals"* and apply it to both metrics.

Success criteria: `propose_min()` returns the expected value for each metric.

In [ ]:
ILLUSTRATIVE_NIGHTLY = {
    "tool_selection_accuracy": [0.90, 0.88, 0.92, 0.89, 0.91],
    "answer_correctness": [0.86, 0.90, 0.84, 0.88, 0.87],
}

In [ ]:
def propose_min(values: list[float]) -> float | None:
    """Gate minimum = mean - 2 * population std dev, rounded DOWN to 2 decimals (None = unsolved)."""
    # TODO: use statistics.fmean / statistics.pstdev and math.floor.
    return None

In [ ]:
import math as _math
import statistics as _statistics

proposals = {m: propose_min(v) for m, v in ILLUSTRATIVE_NIGHTLY.items()}
if any(p is None for p in proposals.values()):
    print("Exercise 2: not solved yet")
else:
    for metric, values in ILLUSTRATIVE_NIGHTLY.items():
        expected = _math.floor((_statistics.fmean(values) - 2 * _statistics.pstdev(values)) * 100 + 1e-9) / 100
        assert abs(proposals[metric] - expected) < 1e-9, f"{metric}: got {proposals[metric]}, expected {expected}"
        assert proposals[metric] < min(values), "a gate above the worst observed run would flap"
    print(f"Exercise 2 passed: {proposals}")

## 5. The gate: `scripts/check_thresholds.py` on a clean and on a regressed run

CI calls the script on `reports/eval_results.json`; its two pure functions, `evaluate()` and
`render()`, work on dictionaries, so we can feed them in-notebook results. The committed baseline
(`reports/baseline/main.json`, created by `make baseline`) adds the regression check when present.

In [ ]:
gate = load_script("check_thresholds")
thresholds = yaml.safe_load((REPO_ROOT / "eval_thresholds.yaml").read_text(encoding="utf-8"))
manifest = json.loads((config.data_dir / "golden" / "manifest.json").read_text(encoding="utf-8"))
baseline_path = REPO_ROOT / "reports" / "baseline" / "main.json"
baseline = json.loads(baseline_path.read_text()) if baseline_path.exists() else None
print("baseline available:", baseline is not None)


def results_payload(cfg: StockroomConfig) -> dict[str, Any]:
    """Same shape as tests/test_trajectory_regression.py writes to reports/eval_results.json."""
    scores = score_suite(cfg)
    return {
        "generated_at": dt.datetime.now(dt.UTC).isoformat(),
        "mode": cfg.mode.value,
        "weaknesses": sorted(cfg.weaknesses),
        "tool_transport": cfg.tool_transport.value,
        "repeats": 1,
        "git_sha": None,
        "dataset": {k: manifest.get(k) for k in ("name", "version", "sha256", "cases")},
        "metrics": aggregate(scores),
        "confidence_intervals": {},
        "cases": [s.model_dump(mode="json") for s in scores],
    }


clean_results = results_payload(config)
rows, failures = gate.evaluate(thresholds, clean_results, promptfoo_clean, baseline)
print("gate failures on the clean run:", failures)
display(Markdown(gate.render(clean_results, rows, failures, promptfoo_clean, baseline)))

In [ ]:
regressed_results = results_payload(config.replace(weaknesses="ambiguous_tool_desc"))
rows, failures = gate.evaluate(thresholds, regressed_results, promptfoo_clean, baseline)
assert failures, "the gate should fail on ambiguous_tool_desc"
display(Markdown(gate.render(regressed_results, rows, failures, promptfoo_clean, baseline)))

The same summary is written to `reports/summary.md`, appended to the GitHub step summary and
posted as a PR comment by `.github/workflows/agent_eval_ci.yml`. With the red-team results from
the unguarded run it fails for a second reason:

In [ ]:
_rows, failures = gate.evaluate(thresholds, clean_results, promptfoo_weak, baseline)
print("\n".join(failures))

## 6. Amazon Bedrock model evaluation with your own inference responses

Bedrock's `CreateEvaluationJob` API runs LLM-as-a-judge evaluations over a JSONL prompt dataset
in S3. With a *precomputed inference source* the responses are supplied in the dataset instead of
being generated by Bedrock, which is what we need for an agent: each line carries the prompt, the
reference answer and the agent's response (`prompt`, `referenceResponse`, `category`,
`modelResponses[{response, modelIdentifier}]`), and the built-in judge metrics score them.

* API reference: https://docs.aws.amazon.com/bedrock/latest/APIReference/API_CreateEvaluationJob.html
* User guide: https://docs.aws.amazon.com/bedrock/latest/userguide/model-evaluation-jobs-management-create.html

The cell below builds the dataset from the golden set and the agent's own answers, builds the
exact request payload and validates it **offline** against the Bedrock service model that ships
with botocore (so the shape is checked without any network call). Submission is a separate cell
with three explicit guards. Note the evaluation job needs its own service role (see
`docs/aws/README.md`); it is not part of the CI role on purpose.

In [ ]:
import tempfile

from botocore import session as botocore_session
from botocore.validate import validate_parameters

SOURCE_ID = "stockroom-harness-" + (config.agent_model_id or "mock").replace(".", "-").replace(":", "-")[:40]
dataset_version = str(manifest["version"]).replace(".", "-")
S3_BUCKET = config.s3_bucket or "<S3_BUCKET>"
S3_PREFIX = config.s3_prefix
JUDGE_MODEL_ID = config.judge_model_id or "<JUDGE_MODEL_ID>"  # from the environment in live mode
EVAL_ROLE_ARN = os.environ.get("EVAL_ROLE_ARN", "<EVAL_ROLE_ARN>")

byoi_rows = []
for case in cases:
    run = harness.run(case.query, case_id=case.id)
    byoi_rows.append(
        {
            "prompt": case.query,
            "referenceResponse": case.reference_answer,
            "category": case.category,
            "modelResponses": [{"response": run.final_answer, "modelIdentifier": SOURCE_ID}],
        }
    )
byoi_dir = Path(tempfile.mkdtemp(prefix="stockroom-byoi-"))
byoi_file = byoi_dir / f"golden_byoi_{dataset_version}.jsonl"
byoi_file.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in byoi_rows) + "\n")
dataset_s3_uri = f"s3://{S3_BUCKET}/{S3_PREFIX}/evaluations/{dataset_version}/{byoi_file.name}"
print(f"{len(byoi_rows)} lines written to {byoi_file}\n")
print(json.dumps(byoi_rows[0], indent=2)[:600])

In [ ]:
job_payload: dict[str, Any] = {
    "jobName": f"stockroom-golden-{dataset_version}-{dt.datetime.now(dt.UTC):%Y%m%d%H%M}",
    "jobDescription": "Stockroom agent answers over the golden set, graded by an LLM judge",
    "roleArn": EVAL_ROLE_ARN,
    "applicationType": "ModelEvaluation",
    "evaluationConfig": {
        "automated": {
            "datasetMetricConfigs": [
                {
                    "taskType": "QuestionAndAnswer",
                    "dataset": {
                        "name": f"stockroom-golden-{dataset_version}",
                        "datasetLocation": {"s3Uri": dataset_s3_uri},
                    },
                    "metricNames": [
                        "Builtin.Correctness",
                        "Builtin.Completeness",
                        "Builtin.Faithfulness",
                        "Builtin.Helpfulness",
                    ],
                }
            ],
            "evaluatorModelConfig": {"bedrockEvaluatorModels": [{"modelIdentifier": JUDGE_MODEL_ID}]},
        }
    },
    "inferenceConfig": {
        "models": [{"precomputedInferenceSource": {"inferenceSourceIdentifier": SOURCE_ID}}]
    },
    "outputDataConfig": {
        "s3Uri": f"s3://{S3_BUCKET}/{S3_PREFIX}/evaluations/{dataset_version}/output/"
    },
}
operation = botocore_session.get_session().get_service_model("bedrock").operation_model(
    "CreateEvaluationJob"
)
validate_parameters(job_payload, operation.input_shape)  # raises ParamValidationError on a bad shape
print("payload matches the CreateEvaluationJob input shape (offline check)\n")
print(json.dumps(job_payload, indent=2))

### Submitting the job (live only, explicit consent)

The job is submitted **only** when all three hold: `STOCKROOM_MODE=live`,
`STOCKROOM_CONFIRM_AWS_SPEND=1` and `EVAL_ROLE_ARN` is set (plus `S3_BUCKET` for the dataset).
Anything else prints what is missing and stops. Submitting incurs Bedrock charges for the judge
model.

In [ ]:
missing = [
    name
    for name, ok in (
        ("STOCKROOM_MODE=live", config.is_live),
        ("STOCKROOM_CONFIRM_AWS_SPEND=1", config.confirm_aws_spend),
        ("EVAL_ROLE_ARN", bool(os.environ.get("EVAL_ROLE_ARN"))),
        ("S3_BUCKET", bool(config.s3_bucket)),
    )
    if not ok
]
if missing:
    print("Not submitting the evaluation job; missing:", ", ".join(missing))
else:
    import boto3

    key = dataset_s3_uri.split(f"s3://{S3_BUCKET}/", 1)[1]
    boto3.client("s3", region_name=config.aws_region).upload_file(str(byoi_file), S3_BUCKET, key)
    bedrock = boto3.client("bedrock", region_name=config.aws_region)
    response = bedrock.create_evaluation_job(**job_payload)
    print("submitted:", response["jobArn"])
    print("poll with bedrock.get_evaluation_job(jobIdentifier=jobArn); results land in outputDataConfig.s3Uri")

## 7. AgentCore Evaluations on a live session (live only)

For agents deployed on AgentCore Runtime, **AgentCore Evaluations** scores session traces with
built-in evaluators (`Builtin.Helpfulness`, `Builtin.ToolSelectionAccuracy`, ...). The on-demand
flow documented at
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-on-demand.html is:

1. the agent runs on AgentCore Runtime instrumented with ADOT, with CloudWatch Transaction Search
   enabled;
2. download the session's spans from CloudWatch Logs with a Logs Insights query;
3. call `bedrock-agentcore.evaluate(evaluatorId=..., evaluationInput={"sessionSpans": [...]})`.

Our mock harness does not run on AgentCore Runtime, so this cell only executes in live mode with
`AGENTCORE_SESSION_ID` set; otherwise it prints the query and the call shape.

In [ ]:
import time

AGENTCORE_SESSION_ID = os.environ.get("AGENTCORE_SESSION_ID")
AGENTCORE_LOG_GROUP = os.environ.get("AGENTCORE_LOG_GROUP", "aws/spans")
# Logs Insights query from the AWS documentation page linked above.
LOGS_INSIGHTS_QUERY = """fields @timestamp, @message
| filter ispresent(scope.name) and ispresent(attributes.session.id)
| filter attributes.session.id = "{session_id}"
| sort @timestamp asc"""


def download_session_spans(session_id: str, log_group: str, region: str) -> list[dict[str, Any]]:
    import boto3

    logs = boto3.client("logs", region_name=region)
    end = dt.datetime.now(dt.UTC)
    start = end - dt.timedelta(minutes=60)
    query_id = logs.start_query(
        logGroupName=log_group,
        startTime=int(start.timestamp()),
        endTime=int(end.timestamp()),
        queryString=LOGS_INSIGHTS_QUERY.format(session_id=session_id),
    )["queryId"]
    while (result := logs.get_query_results(queryId=query_id))["status"] not in ("Complete", "Failed"):
        time.sleep(1)
    if result["status"] == "Failed":
        raise RuntimeError("Logs Insights query failed")
    return [
        json.loads(f["value"])
        for row in result["results"]
        for f in row
        if f["field"] == "@message" and f["value"].strip().startswith("{")
    ]


if config.is_live and AGENTCORE_SESSION_ID:
    import boto3

    session_spans = download_session_spans(AGENTCORE_SESSION_ID, AGENTCORE_LOG_GROUP, config.aws_region)
    agentcore = boto3.client("bedrock-agentcore", region_name=config.aws_region)
    response = agentcore.evaluate(
        evaluatorId="Builtin.Helpfulness", evaluationInput={"sessionSpans": session_spans}
    )
    for result in response["evaluationResults"]:
        print(result.get("evaluatorId"), result.get("value"), result.get("label"), result.get("errorCode"))
else:
    print("AgentCore Evaluations skipped (needs STOCKROOM_MODE=live and AGENTCORE_SESSION_ID).\n")
    print("Logs Insights query:\n" + LOGS_INSIGHTS_QUERY.format(session_id="<session-id>"))
    print('\ncall shape: bedrock-agentcore.evaluate(evaluatorId="Builtin.Helpfulness", '
          'evaluationInput={"sessionSpans": [...]})')

## 8. How CI reaches AWS: GitHub OIDC

Two workflows live in `.github/workflows/`:

* **`agent_eval_ci.yml`** — the PR gate. Mock mode, no credentials, no network: lint, data
  validation, unit tests, the golden suite over the MCP server, the Promptfoo suite, these
  notebooks, then `check_thresholds.py`. It requests no AWS permissions at all.
* **`agent_eval_nightly.yml`** — live mode, report only, started by hand (`workflow_dispatch`; the
  repo keeps no cron so Bedrock spend happens only when someone asks for a run). It declares `permissions: id-token:
  write`, and `aws-actions/configure-aws-credentials` exchanges the job's GitHub OIDC token for
  short-lived credentials of an IAM role (`AWS_OIDC_ROLE_ARN`, a repository variable). The role's
  trust policy (`docs/aws/oidc_trust_policy.json`) only accepts tokens whose `sub` is this
  repository's `main` branch, and its permissions policy (`docs/aws/iam_policy.json`) allows
  invoking only the two configured models and one S3 prefix. No long-lived access key exists
  anywhere.

In [ ]:
def show(path: Path, needles: tuple[str, ...]) -> None:
    print(f"--- {path.relative_to(REPO_ROOT)}")
    for line in path.read_text(encoding="utf-8").splitlines():
        if any(n in line for n in needles):
            print(line.rstrip())
    print()


workflows = REPO_ROOT / ".github" / "workflows"
if workflows.exists():
    show(workflows / "agent_eval_ci.yml", ("permissions:", "contents:", "pull-requests:", "STOCKROOM_MODE"))
    show(
        workflows / "agent_eval_nightly.yml",
        ("permissions:", "id-token:", "configure-aws-credentials", "role-to-assume", "STOCKROOM_MODE", "--no-gate"),
    )
    print((REPO_ROOT / "docs" / "aws" / "oidc_trust_policy.json").read_text(encoding="utf-8"))
else:
    print("workflow files not found next to this checkout; see .github/workflows in the repository")

### Exercise 3 — a synthetic-case acceptance rule

Turn the candidate table from section 2 into a rule. `accept_candidate(case, scores)` must accept
a candidate only when **all** of these hold:

1. the judge passed and the tool selection score is 1.0;
2. the run terminated `COMPLETED`;
3. the case has at least one expected fact;
4. the query is not already in the golden set.

Success criteria: the two deliberately bad candidates (`S011` duplicate, `S012` mislabelled) are
rejected and every other candidate is accepted.

In [ ]:
def accept_candidate(case: GoldenCase, scores: CaseScores) -> bool:
    """Return True when the synthetic case may join the golden set."""
    # TODO: implement the four rules above (scores.judge_passed, scores.tool_selection,
    #       scores.termination_reason, case.expected_facts, golden_queries).
    return True

In [ ]:
decisions = {c.id: accept_candidate(c, s) for c, _r, s in candidate_scores}
if all(decisions.values()):
    print("Exercise 3: not solved yet (every candidate is accepted)")
else:
    rejected = sorted(cid for cid, ok in decisions.items() if not ok)
    assert rejected == ["S011", "S012"], f"rejected {rejected}, expected ['S011', 'S012']"
    print(f"Exercise 3 passed: accepted {len(decisions) - len(rejected)}, rejected {rejected}")

## Wrap-up

* Offline evals gate PRs; nightly live runs report confidence intervals; online evaluators score
  real sessions. Each catches something the others cannot.
* Grow the golden set with a teacher, but keep labels data-derived and let a calibrated judge plus
  explicit rules decide what gets in.
* Thresholds come from measured variance; the gate script turns them into a readable PR comment.
* Bedrock model evaluation and AgentCore Evaluations are the managed counterparts of the judge
  and the trace evaluators you built; both are called only with explicit, auditable consent.